# 01 — Data Preparation for SFT Intent Classifier

This notebook prepares **Veridian Systems** historical IT ticket data for fine-tuning
a lightweight intent classifier on Together.ai.
It converts raw tickets to the SFT JSONL format and splits the data into
train / val / test sets. The split files are read directly by `02_train_classifier.ipynb`.

In [1]:
!uv add mistralai pandas scikit-learn python-dotenv rich

Resolved 151 packages in 0.74ms
Audited 145 packages in 1ms


In [2]:
import json
import os
import uuid
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR        = Path("data")
RAW_FILE        = DATA_DIR / "raw_tickets.json"
SYNTHETIC_FILE  = DATA_DIR / "raw_tickets_synthetic.json"
TRAIN_FILE      = DATA_DIR / "train.jsonl"
VAL_FILE        = DATA_DIR / "val.jsonl"
TEST_FILE       = DATA_DIR / "test.jsonl"

# ── SFT classifier system prompt ──────────────────────────────────────────
CLASSIFIER_SYSTEM_PROMPT = (
    "Classify the IT support request into exactly one of the following categories: "
    "access_request, security_incident, hardware_issue, software_issue, onboarding, "
    "payments_incident, expense_request, general_question. "
    "Disambiguation rules: "
    "onboarding vs access_request: use onboarding when a new hire in their first 1-2 weeks cannot access a tool as part of initial setup; "
    "use access_request when an established employee requests a permission change, new project access, role transfer, contractor provisioning, or offboarding — "
    "even if they recently joined, once initial setup is complete any further access change is access_request. "
    "payments_incident vs security_incident: use payments_incident when prod-payments has a service failure or degradation (HTTP errors, timeouts, crashes, OOM, duplicate charges, cert expiry, deployment failures); "
    "use security_incident when the trigger is a threat or credential exposure — phishing, stolen device, leaked API key, unauthorized access, malware — "
    "even if the exposed credential or affected system belongs to prod-payments. "
    "expense_request vs general_question: use expense_request when the person has a specific purchase in mind and is seeking approval or process guidance for it; "
    "use general_question when asking about a policy or process with no specific purchase stated. "
    "software_issue vs access_request: use software_issue when the person has access but something is broken, misconfigured, or expired; "
    "use access_request when they need access granted for the first time or revoked entirely. "
    "onboarding vs general_question: use onboarding when a new hire cannot access a specific tool needed for their role; "
    "use general_question for policy or how-to questions even if the asker is new. "
    "hardware_issue vs onboarding: use onboarding when a new hire's device is not yet set up or enrolled; "
    "use hardware_issue when a device is malfunctioning for an existing employee or after initial setup is complete. "
    "Respond with only the category label, nothing else."
)

# ── All valid intent labels (order matters for reproducibility) ───────────
INTENT_LABELS = [
    "access_request",
    "security_incident",
    "hardware_issue",
    "software_issue",
    "onboarding",
    "payments_incident",
    "expense_request",
    "general_question",
]

RANDOM_STATE = 42

## 1. Overview

### What this notebook does

1. Loads `data/raw_tickets.json` — 122 labelled Veridian IT support tickets across
   8 intent categories.
2. Converts each ticket to the **SFT JSONL format**:
   ```json
   {"messages": [
     {"role": "system", "content": "<classifier system prompt>"},
     {"role": "user",   "content": "<ticket text>"},
     {"role": "assistant", "content": "<intent label>"}
   ]}
   ```
   The assistant turn contains only the intent label — the model learns to output
   exactly one of the 8 labels given a ticket description.
3. Splits the data **70 % train / 15 % val / 15 % test**, stratified by intent
   so every class is proportionally represented in all three splits.
4. Saves `data/train.jsonl`, `data/val.jsonl`, and `data/test.jsonl`.
   > **Do not regenerate these files after the classifier has been trained** —
   > changing the split would invalidate test-set evaluation results.
5. Uploads `train.jsonl` and `val.jsonl` to the Mistral Files API and
   prints the file IDs needed for `02_train_classifier.ipynb`.

### Intent categories

| Label | Description |
|---|---|
| `access_request` | Nexus, Prism, GitHub, AWS, Okta provisioning / offboarding |
| `security_incident` | SEV1/SEV2/SEV3 events — phishing, stolen device, suspicious login |
| `hardware_issue` | MacBook M3 MDM, monitors, docks, peripherals, battery |
| `software_issue` | Okta MFA, Slack, Zoom, IDE licences, Docker, Nexus auth |
| `onboarding` | Day-1 setup, Helix on-call, Prism access, badge, VPN |
| `payments_incident` | prod-payments degradation — always P1, paged via Helix |
| `expense_request` | Home office budget, ergonomics, L&D, Adobe, internet stipend |
| `general_question` | Policy queries, Jira routing, WiFi, HR redirects |

## 2. Load and explore raw tickets

In [7]:
with open(RAW_FILE) as f:
    raw_tickets = json.load(f)

# Merge synthetic tickets if they have been generated
SYNTHETIC_FILE = DATA_DIR / "raw_tickets_synthetic.json"
if SYNTHETIC_FILE.exists():
    with open(SYNTHETIC_FILE) as f:
        synthetic_tickets = json.load(f)
    raw_tickets = raw_tickets + synthetic_tickets
    console.print(f"[dim]Merged {len(synthetic_tickets)} synthetic tickets from {SYNTHETIC_FILE.name}[/dim]")

df = pd.DataFrame(raw_tickets)
console.print(f"Loaded [bold]{len(df)}[/bold] tickets across [bold]{df['intent'].nunique()}[/bold] intent categories\n")

# Validate: every ticket must have exactly the fields we need
required_fields = {"id", "text", "intent", "priority"}
missing = [t["id"] for t in raw_tickets if not required_fields.issubset(t.keys())]
if missing:
    raise ValueError(f"Tickets missing required fields: {missing}")

# Validate: all intents are from the known set
unknown_intents = set(df["intent"]) - set(INTENT_LABELS)
if unknown_intents:
    raise ValueError(f"Unknown intent labels in data: {unknown_intents}")

console.print("[green]✓[/green] All fields present and intent labels valid\n")
df[["id", "intent", "priority", "text"]].head(8)

Merged 300 synthetic tickets from raw_tickets_synthetic.json

Loaded 422 tickets across 8 intent categories

✓ All fields present and intent labels valid

,id,intent,priority,text
0,INC-0001,access_request,P2,"Hi IT, I need read/write access to Nexus for t..."
1,INC-0002,access_request,P2,I need Prism access to run our Q3 cohort analy...
2,INC-0003,access_request,P2,We have a contractor from TechFlow starting Mo...
3,INC-0004,access_request,P2,I'm on SRE and need read access to the prod AW...
4,INC-0005,access_request,P1,Please remove all system access for Raj Patel ...
5,INC-0006,access_request,P3,Can I get added to the INFRA Jira project? I'm...
6,INC-0007,access_request,P3,I transferred to the Austin team last month bu...
7,INC-0008,access_request,P3,I need Okta admin access. I manage several Saa...


In [8]:
dist = df["intent"].value_counts().reindex(INTENT_LABELS)

table = Table(title="Class Distribution — raw_tickets.json", show_header=True, header_style="bold")
table.add_column("Intent", style="cyan", min_width=22)
table.add_column("Count", justify="right", style="magenta")
table.add_column("Share", justify="right")
table.add_column("Bar", min_width=20)

for intent, count in dist.items():
    bar = "█" * count
    table.add_row(intent, str(count), f"{count / len(df):.0%}", bar)

console.print(table)
console.print(
    "[dim]Note: 12–16 examples per class provides good coverage for the pipeline. "
    "For production, aim for ≥ 50 examples per class.[/dim]"
)

                                Class Distribution — raw_tickets.json                                
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Intent                 ┃ Count ┃ Share ┃ Bar                                                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ access_request         │    50 │   12% │ ██████████████████████████████████████████████████       │
│ security_incident      │    54 │   13% │ ██████████████████████████████████████████████████████   │
│ hardware_issue         │    56 │   13% │ ████████████████████████████████████████████████████████ │
│ software_issue         │    53 │   13% │ █████████████████████████████████████████████████████    │
│ onboarding             │    54 │   13% │ ██████████████████████████████████████████████████████   │
│ payments_incident      │    49 │   12% │ █████████████████████████████████████████████████        │
│ expense_request        │    52 │   12% │ ████████████████████████████████████████████████████     │
│ general_question       │    54 │   13% │ ██████████████████████████████████████████████████████   │
└────────────────────────┴───────┴───────┴──────────────────────────────────────────────────────────┘

Note: 12–16 examples per class provides good coverage for the pipeline. For production, aim for ≥ 50 examples per 
class.

## 3. Generate synthetic tickets (optional — recommended)

122 examples (~15 per class) provides a solid fine-tuning foundation but
synthetic data still adds valuable linguistic variety. This section generates
**40 additional synthetic tickets per class** (320 new examples) using
`mistral-large-latest`, bringing the total to ~442 examples (~55 per class).

The synthetic tickets are saved to `data/raw_tickets_synthetic.json` and merged
with the originals before splitting. `raw_tickets.json` is never modified.

Skip this cell if you want to use the 122-example raw dataset only.


In [6]:
try:
    from google.colab import userdata
    MISTRAL_API_KEY = userdata.get("MISTRAL_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

from mistralai.client import Mistral

N_PER_CLASS = 40   # synthetic tickets to generate per intent class

_CLASS_DESCRIPTIONS = {
    "access_request":    "account provisioning, permission changes, or access removal — e.g. Nexus artifact repo, Prism data warehouse, GitHub org, AWS IAM, Okta group, offboarding",
    "security_incident": "suspected breach, phishing email, ransomware, stolen/lost device, suspicious login, unauthorised data access — any SEV-tier security event",
    "hardware_issue":    "broken or malfunctioning physical equipment — MacBook, monitor, dock, keyboard, mouse, webcam, headset, battery, charging cable, printer",
    "software_issue":    "software install requests, licence renewals, version upgrades, SSO/MFA problems — Okta Verify, Slack, Zoom, JetBrains, Docker, VS Code, Nexus auth",
    "onboarding":        "new-hire day-1 setup — MDM enrolment, Slack/Okta account creation, Helix on-call rotation, Prism read access, VPN profile, badge, laptop provisioning",
    "payments_incident": "any degradation or error touching the prod-payments service — webhook delays, 5xx errors, duplicate charges, Braintree/Stripe failures, P99 latency spikes",
    "expense_request":   "reimbursement or budget approval — home office equipment, ergonomic desk, L&D course, conference ticket, internet stipend, software subscription, doctor-recommended gear",
    "general_question":  "policy questions, how-to queries, SLA questions, Jira routing questions, WiFi help, HR redirects — requests that need an answer, not a ticket",
}

_GENERATION_PROMPT = """You are generating realistic IT support tickets for Veridian Systems, a B2B SaaS company (~800 employees).

Generate {n} distinct IT support ticket texts for the intent category: **{intent}**
Category description: {description}

Rules:
- Each ticket must be 1–4 sentences, written by an employee (first person, informal)
- Vary the writing style: some terse, some detailed, some urgent
- Use Veridian-specific internal names where natural: Nexus (artifact repo), Prism (data warehouse), Helix (on-call tool), prod-payments (critical payment service)
- Do NOT include the intent label in the ticket text
- Do NOT number the tickets or add any prefixes — output one ticket per line, nothing else
- Each ticket must be on a single line with no blank lines between them

Output exactly {n} lines."""

mistral_client = Mistral(api_key=MISTRAL_API_KEY)
synthetic_tickets: list[dict] = []

for intent, description in _CLASS_DESCRIPTIONS.items():
    console.print(f"Generating [bold]{N_PER_CLASS}[/bold] synthetic tickets for [cyan]{intent}[/cyan]…")

    response = mistral_client.chat.complete(
        model="mistral-large-latest",
        messages=[{
            "role": "user",
            "content": _GENERATION_PROMPT.format(
                n=N_PER_CLASS,
                intent=intent,
                description=description,
            ),
        }],
        temperature=0.9,
        max_tokens=2000,
    )

    raw_text = response.choices[0].message.content.strip()
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]

    if len(lines) != N_PER_CLASS:
        console.print(f"  [yellow]Warning: expected {N_PER_CLASS} lines, got {len(lines)} — using what we got[/yellow]")

    for line in lines:
        synthetic_tickets.append({
            "id":       f"SYN-{uuid.uuid4().hex[:6].upper()}",
            "text":     line,
            "intent":   intent,
            "priority": "P2",
        })

SYNTHETIC_FILE.write_text(json.dumps(synthetic_tickets, indent=2))
console.print(Panel(
    f"Generated [bold]{len(synthetic_tickets)}[/bold] synthetic tickets\n"
    f"Saved → [cyan]{SYNTHETIC_FILE}[/cyan]\n"
    f"[dim]Re-run cell-5 (load) to merge before splitting.[/dim]",
    title="[green]Synthetic data ready[/green]",
    border_style="green",
))

Generating 40 synthetic tickets for access_request…

Warning: expected 40 lines, got 34 — using what we got

Generating 40 synthetic tickets for security_incident…

Warning: expected 40 lines, got 38 — using what we got

Generating 40 synthetic tickets for hardware_issue…

...

Generating 40 synthetic tickets for expense_request…

Warning: expected 40 lines, got 37 — using what we got

Generating 40 synthetic tickets for general_question…

Warning: expected 40 lines, got 39 — using what we got

╭───────────────────────────────────────────── Synthetic data ready ──────────────────────────────────────────────╮
│ Generated 300 synthetic tickets                                                                                 │
│ Saved → data/raw_tickets_synthetic.json                                                                         │
│ Re-run cell-5 (load) to merge before splitting.                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. Convert to SFT JSONL format

Each training example is a JSON object with a `messages` list containing three turns:
a fixed system prompt, the ticket text as the user turn, and the intent label as the
assistant turn.

```json
{"messages": [
  {"role": "system",    "content": "<CLASSIFIER_SYSTEM_PROMPT>"},
  {"role": "user",      "content": "<ticket text>"},
  {"role": "assistant", "content": "<intent label>"}
]}
```

The assistant turn contains **only** the intent label — no punctuation, no explanation.
The fine-tuned model learns to reproduce this output given any new ticket.

At inference time `AdaptedAgent._classify()` sends the same system prompt and the
user's message; the model's response is parsed directly as the intent label.

In [9]:
def ticket_to_example(ticket: dict) -> dict:
    """Convert a raw ticket dict to an SFT training example."""
    return {
        "messages": [
            {"role": "system",    "content": CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user",      "content": ticket["text"]},
            {"role": "assistant", "content": ticket["intent"]},
        ]
    }


examples = [ticket_to_example(t) for t in raw_tickets]

console.print(f"Converted [bold]{len(examples)}[/bold] tickets to SFT format\n")

console.print("[bold]3 example records:[/bold]")
for ex in examples[:3]:
    msgs = ex["messages"]
    console.print(Panel(
        f"[bold cyan]intent:[/bold cyan] {msgs[-1]['content']}\n"
        f"[bold]text:[/bold] {msgs[1]['content']}",
        expand=False,
    ))

Converted 422 tickets to SFT format

3 example records:

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ intent: access_request                                                                                          │
│ text: Hi IT, I need read/write access to Nexus for the payments-service namespace. I just joined the backend    │
│ team and can't pull our internal libs to build locally. My GitHub handle is @lena-okoye.                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ intent: access_request                                                                                          │
│ text: I need Prism access to run our Q3 cohort analysis. I should have analyst-tier read access but the Prism   │
│ login page is rejecting my Okta credentials with 'access denied'. Manager is Daniel Wu.                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ intent: access_request                                                                                          │
│ text: We have a contractor from TechFlow starting Monday to help on the API migration. They need GitHub org     │
│ access (read on all repos, write on api-gateway) and Jira access for the APIMIG project. Contract end date is   │
│ Dec 31.                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 4. Train / validation / test split

Splits: **70 % train · 15 % val · 15 % test**, stratified by intent.

Stratification ensures every intent class appears in all three splits in roughly
the same proportion as in the full dataset — important when some classes
(e.g. `payments_incident`) have fewer examples than others.

The split is done in two passes:
1. Separate 70 % train from 30 % remainder (stratified).
2. Split the 30 % remainder 50/50 into val and test (stratified).

In [10]:
all_labels = [ex["messages"][-1]["content"] for ex in examples]

# Pass 1: 70% train, 30% temp
train_examples, temp_examples = train_test_split(
    examples,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=all_labels,
)

# Pass 2: split temp 50/50 → val (15%) and test (15%)
temp_labels = [ex["messages"][-1]["content"] for ex in temp_examples]
val_examples, test_examples = train_test_split(
    temp_examples,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_labels,
)

console.print(
    f"Split complete: "
    f"train=[bold]{len(train_examples)}[/bold]  "
    f"val=[bold]{len(val_examples)}[/bold]  "
    f"test=[bold]{len(test_examples)}[/bold]"
)

Split complete: train=295  val=63  test=64

In [11]:
def count_by_label(split: list[dict]) -> dict[str, int]:
    counts: dict[str, int] = {label: 0 for label in INTENT_LABELS}
    for ex in split:
        counts[ex["messages"][-1]["content"]] += 1
    return counts


train_counts = count_by_label(train_examples)
val_counts   = count_by_label(val_examples)
test_counts  = count_by_label(test_examples)

table = Table(
    title=f"Examples per split per category  (total {len(examples)})",
    show_header=True,
    header_style="bold",
)
table.add_column("Intent", style="cyan", min_width=22)
table.add_column("Train",  justify="right", style="green")
table.add_column("Val",    justify="right", style="yellow")
table.add_column("Test",   justify="right", style="magenta")
table.add_column("Total",  justify="right")

for label in INTENT_LABELS:
    tr, v, te = train_counts[label], val_counts[label], test_counts[label]
    table.add_row(label, str(tr), str(v), str(te), str(tr + v + te))

table.add_section()
table.add_row(
    "[bold]TOTAL[/bold]",
    f"[bold]{len(train_examples)}[/bold]",
    f"[bold]{len(val_examples)}[/bold]",
    f"[bold]{len(test_examples)}[/bold]",
    f"[bold]{len(examples)}[/bold]",
)

console.print(table)

     Examples per split per category  (total 422)      
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━┳━━━━━━┳━━━━━━━┓
┃ Intent                 ┃ Train ┃ Val ┃ Test ┃ Total ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━╇━━━━━━╇━━━━━━━┩
│ access_request         │    35 │   8 │    7 │    50 │
│ security_incident      │    38 │   8 │    8 │    54 │
│ hardware_issue         │    39 │   8 │    9 │    56 │
│ software_issue         │    37 │   8 │    8 │    53 │
│ onboarding             │    38 │   8 │    8 │    54 │
│ payments_incident      │    34 │   7 │    8 │    49 │
│ expense_request        │    36 │   8 │    8 │    52 │
│ general_question       │    38 │   8 │    8 │    54 │
├────────────────────────┼───────┼─────┼──────┼───────┤
│ TOTAL                  │   295 │  63 │   64 │   422 │
└────────────────────────┴───────┴─────┴──────┴───────┘

## 5. Save splits

In [12]:
def save_jsonl(split: list[dict], path: Path) -> None:
    """Write a list of dicts to a JSONL file, one record per line."""
    with open(path, "w") as f:
        for record in split:
            f.write(json.dumps(record) + "\n")
    console.print(f"  Saved [bold]{len(split):>3}[/bold] examples → [cyan]{path}[/cyan]")


console.print("[bold]Writing JSONL files:[/bold]")
save_jsonl(train_examples, TRAIN_FILE)
save_jsonl(val_examples,   VAL_FILE)
save_jsonl(test_examples,  TEST_FILE)

# ── Spot-check the first line of each file ────────────────────────────────
console.print("\n[bold]Format check (first record of each file):[/bold]")
for path in (TRAIN_FILE, VAL_FILE, TEST_FILE):
    with open(path) as f:
        first = json.loads(f.readline())
    msgs = first["messages"]
    console.print(
        f"  [cyan]{path.name}[/cyan]  "
        f"intent=[green]{msgs[-1]['content']}[/green]  "
        f"text=[dim]{msgs[1]['content'][:72]}…[/dim]"
    )

Writing JSONL files:

Saved 295 examples → data/train.jsonl

Saved  63 examples → data/val.jsonl

Saved  64 examples → data/test.jsonl

Format check (first record of each file):

train.jsonl  intent=payments_incident  text=I tried to void a transaction and got a 503 error—is the payment 
service…

val.jsonl  intent=general_question  text=Is there a list of approved software we can install? I want to make 
sure…

test.jsonl  intent=payments_incident  text=Helix is showing a critical alert for prod-payments with a 50% error 
rat…

## 6. Forge context

### How this step relates to a production Forge deployment

In a production **Forge** deployment, the domain-adaptation step may not be
a supervised classification fine-tune on labelled tickets.
Instead it could be **continued pre-training on unlabelled enterprise text** —
ticket bodies, internal wiki pages, runbook documents, architecture decision
records, and Slack incident threads — allowing the base model to absorb
a larger amount of Veridian-specific terminology (Nexus, Prism, Helix,
prod-payments, SEV tiers) at the level of the full model weights.

**SFT fine-tuning is a targeted, supervised variant of the same fine-tuning
infrastructure.** Rather than adapting the entire model to a new text domain,
it fine-tunes the model weights to map input text to a discrete label — using
the same underlying training pipeline as Forge.

| | SFT fine-tuning (this demo) | Forge continued pre-training |
|---|---|---|
| **Input data** | Labelled `{messages}` examples | Unlabelled text corpus |
| **Training signal** | Cross-entropy on known labels | Next-token prediction |
| **What adapts** | Full model weights (small base) | Full model weights |
| **Output** | Intent classifier model ID | Domain-adapted base model |
| **Data residency** | Together.ai | Forge (on-prem / private cloud) |
| **Typical data size** | Tens to thousands of examples | Gigabytes of text |

The Python SDK code in this notebook and in `02_train_classifier.ipynb` would
require **only a `server_url` change** to target a Forge endpoint instead of
Together — the file upload, job creation, and polling logic is identical.